In [1]:
A = [
    [0.15, 2.11, 3.75, 8.14],
    [0.64, 1.21, 2.05, -0.99],
    [3.21, 1.53, -1.04, -3.18],
    [0.77, 1.22, 1.18, 2.25]
]

b = [-6.35, 2.47, 3.82, -1.52]

eps = 1e-4

In [2]:
# ===============================
# 1. Метод Гаусса
# ===============================
def gauss(A, b):
    n = len(A)

    # создаем копии
    A = [row[:] for row in A]
    b = b[:]

    # прямой ход
    for k in range(n):
        # выбор главного элемента
        max_row = k
        for i in range(k + 1, n):
            if abs(A[i][k]) > abs(A[max_row][k]):
                max_row = i

        A[k], A[max_row] = A[max_row], A[k]
        b[k], b[max_row] = b[max_row], b[k]

        for i in range(k + 1, n):
            factor = A[i][k] / A[k][k]
            for j in range(k, n):
                A[i][j] -= factor * A[k][j]
            b[i] -= factor * b[k]

    # обратный ход
    x = [0] * n
    for i in range(n - 1, -1, -1):
        s = b[i]
        for j in range(i + 1, n):
            s -= A[i][j] * x[j]
        x[i] = s / A[i][i]

    return x

In [3]:
# ===============================
# 2. Нормальные уравнения
# ===============================
def transpose(A):
    return [[A[j][i] for j in range(len(A))] for i in range(len(A[0]))]


def matmul(A, B):
    result = [[0]*len(B[0]) for _ in range(len(A))]
    for i in range(len(A)):
        for j in range(len(B[0])):
            for k in range(len(B)):
                result[i][j] += A[i][k] * B[k][j]
    return result


def matvec(A, x):
    return [sum(A[i][j]*x[j] for j in range(len(x)))
            for i in range(len(A))]


In [4]:
MAX_ITERS = 2**17  # 131072

def seidel_by_residual(C, d, A_orig, b_orig, eps=1e-4):
    n = len(d)
    x = [0.0] * n

    for k in range(1, MAX_ITERS + 1):
        x_new = x[:]

        # шаг Зейделя
        for i in range(n):
            s = d[i]

            # j < i
            for j in range(i):
                s -= C[i][j] * x_new[j]

            # j > i
            for j in range(i + 1, n):
                s -= C[i][j] * x[j]

            x_new[i] = s / C[i][i]

        # невязка исходной системы
        r = matvec(A_orig, x_new)
        for i in range(n):
            r[i] -= b_orig[i]

        res = max(abs(val) for val in r)

        if res < eps:
            return x_new, k, res

        x = x_new

    return x, MAX_ITERS, res


In [5]:
x_gauss = gauss(A, b)
print("Метод Гаусса:")
print([round(v, 6) for v in x_gauss])

AT = transpose(A)
C = matmul(AT, A)
d = matvec(AT, b)

x_seidel, iters, res = seidel_by_residual(C, d, A, b, eps=1e-4)

print("Метод Зейделя:")
print([round(v, 6) for v in x_seidel])
print("Количество итераций:", iters)
print("Невязка ||Ax-b||_inf =", res)


Метод Гаусса:
[1.0, -1.0, 1.0, -1.0]
Метод Зейделя:
[0.999126, -0.998708, 0.999471, -1.00008]
Количество итераций: 8796
Невязка ||Ax-b||_inf = 9.997180923893012e-05


In [ ]:
def seidel(C, d, A_orig, b_orig, eps):
    n = len(d)
    x = [0.0]*n

    for k in range(2**17):
        x_new = x[:]

        for i in range(n):
            s = d[i]
            # j < i
            for j in range(i):
                s -= C[i][j] * x_new[j]
            # j > i
            for j in range(i+1, n):
                s -= C[i][j] * x[j]
            x_new[i] = s / C[i][i]

        # критерий по невязке исходной системы
        r = matvec(A_orig, x_new)
        r = [r[i] - b_orig[i] for i in range(n)]
        res = max(abs(v) for v in r)

        if res < eps:
            return x_new, k+1, res

        x = x_new

    return x, 2**17, res


In [2]:
import numpy as np

# исходная система (принудительно 2 знака)
A = np.round(np.array([
    [0.15, 2.11, 3.75, 8.14],
    [0.64, 1.21, 2.05, -0.99],
    [3.21, 1.53, -1.04, -3.18],
    [0.77, 1.22, 1.18, 2.25]
]), 2)

b = np.round(np.array([-6.35, 2.47, 3.82, -1.52]), 2)

eps = 1e-2


x_gauss = np.linalg.solve(A, b)

print("Метод Гаусса:")
print(np.round(x_gauss, 2))

C = A.T @ A
d = A.T @ b

def seidel(C, d, A_orig, b_orig, eps=1e-4, max_iter=2**17):
    n = len(d)
    x = np.zeros(n)

    for k in range(max_iter):
        x_new = x.copy()

        for i in range(n):
            s1 = np.dot(C[i, :i], x_new[:i])
            s2 = np.dot(C[i, i+1:], x[i+1:])
            x_new[i] = (d[i] - s1 - s2) / C[i, i]

        # критерий по невязке исходной системы
        residual = A_orig @ x_new - b_orig
        if np.linalg.norm(residual, ord=np.inf) < eps:
            return x_new, k+1

        x = x_new

    return x, max_iter

x_seidel, iters = seidel(C, d, A, b, eps)

print("Метод Зейделя:")
print(np.round(x_seidel, 2))
print("Итераций:", iters)

res = np.linalg.norm(A @ x_seidel - b, ord=np.inf)
print("Невязка ||Ax-b||_∞ =", res)




Метод Гаусса:
[ 1. -1.  1. -1.]
Метод Зейделя:
[ 0.91 -0.87  0.95 -1.01]
Итераций: 2792
Невязка ||Ax-b||_∞ = 0.00999983643232305


In [8]:
eps = 1e-4


def seidel(C, d, A_orig, b_orig, eps=1e-4, max_iter=2**17):
    n = len(d)
    x = np.zeros(n)

    history = []  # сюда будем сохранять последние 5 итераций

    for k in range(1, max_iter + 1):
        x_old = x.copy()
        x_new = x.copy()

        for i in range(n):
            s1 = np.dot(C[i, :i], x_new[:i])
            s2 = np.dot(C[i, i+1:], x_old[i+1:])
            x_new[i] = (d[i] - s1 - s2) / C[i, i]

        # сохраняем в историю
        history.append((k, x_new.copy()))
        if len(history) > 5:
            history.pop(0)

        # невязка исходной системы
        residual = A_orig @ x_new - b_orig
        res_norm = np.linalg.norm(residual, ord=np.inf)

        if res_norm < eps:
            return x_new, k, res_norm, history

        x = x_new

    return x, max_iter, res_norm, history

x_seidel, iters, res_norm, history = seidel(C, d, A, b, eps)

print("Метод Зейделя")
print("Итераций:", iters)
print("Невязка ||Ax-b||_∞ =", res_norm)

print("\nПоследние 5 итераций:")
for k, vec in history:
    print(f"k = {k}: {np.round(vec, 6)}")


Метод Зейделя
Итераций: 8796
Невязка ||Ax-b||_∞ = 9.997180919896209e-05

Последние 5 итераций:
k = 8792: [ 0.999123 -0.998704  0.99947  -1.00008 ]
k = 8793: [ 0.999124 -0.998705  0.99947  -1.00008 ]
k = 8794: [ 0.999125 -0.998706  0.99947  -1.00008 ]
k = 8795: [ 0.999125 -0.998707  0.999471 -1.00008 ]
k = 8796: [ 0.999126 -0.998708  0.999471 -1.00008 ]
